# Getting started with Megamicros v4.0
® *Copyright Bimea 2024-2026*

Welcome to Megamicros v4.0! This notebook introduces the new multi-source architecture while maintaining full backward compatibility with v3.x.

## What's New in v4.0

- **Multi-source system**: USB hardware, H5 files, random generator, or WebSocket
- **Automatic source selection**: The library detects the best source automatically
- **Testing without hardware**: Use `RandomDataSource` for development
- **Type-safe configuration**: Use `AcquisitionConfig` for better validation
- **100% backward compatible**: All v3.x code still works!

Let's explore the basics of the `Megamicros` class.

In [ ]:
import time
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt

# v4.0 imports
import megamicros
from megamicros import Megamicros, AcquisitionConfig
from megamicros.log import log

# Set log level to INFO to get some information
# Available levels: DEBUG, INFO, WARNING, ERROR, CRITICAL
log.setLevel("INFO")

print(f"Megamicros version: {megamicros.__version__}")

## Create an Antenna Object

In v4.0, the `Megamicros` class automatically selects the appropriate data source:

- If you have USB hardware connected → **UsbDataSource**
- If you specify a file → **H5DataSource** 
- If no hardware is found → **RandomDataSource** (perfect for testing!)

Let's create an antenna and see what source is selected.

In [ ]:
# Create antenna (auto-detects source)
antenna = Megamicros()

# Get information about the antenna
info = antenna.infos
pprint(info)

print(f"\n✓ Source type: {info['source_type']}")
print(f"✓ Available MEMS: {len(antenna.available_mems)}")

## Basic Acquisition

Let's activate the first 8 MEMS and acquire 1 second of data.

**v4.0 pattern**: `run()` starts acquisition (non-blocking), `wait()` blocks until complete.

**Important**: 
- `wait()` does NOT empty the queue - frames remain available for iteration!
- **NEW**: Frames ACCUMULATE between multiple `run()` calls (unless queue_size changes)
- Use `clear_queue()` when you want a fresh start

In [ ]:
# v4.0: All parameters in run() method
antenna.run(
    mems=list(range(8)),    # Activate first 8 MEMS
    duration=1,              # 1 second acquisition
    sampling_frequency=44100,
    frame_length=1024
)

# Wait for acquisition to complete
antenna.wait()

print(f"✓ Acquisition complete!")
print(f"✓ Frames in queue: {antenna.queue_content}")

## Retrieving Data by Iteration

You can iterate through the `antenna` object to get frames.  
**Important**: Each frame is yielded once - iteration empties the queue!

In [ ]:
# Check queue content before iteration
print(f"Queue content before: {antenna.queue_content} frames")

# Retrieve data from the queue
frame_count = 0
for frame in antenna:
    print(f"  Frame {frame_count}: shape={frame.shape}, dtype={frame.dtype}")
    frame_count += 1
    if frame_count >= 3:  # Show only first 3 frames
        print(f"  ... (and {antenna.queue_content} more)")
        break

print(f"\n✓ Retrieved {frame_count} frames")

## Clearing the Queue Explicitly

If you want to discard frames without processing them, use the `clear_queue()` method.

**Why is this needed?** Since v4.0, frames **accumulate** between runs! This gives you flexibility:
- **Accumulate data**: Run multiple acquisitions without clearing → combine later
- **Fresh start**: Call `clear_queue()` before `run()` → start from zero

This is useful when experimenting or when you only care about the next acquisition.

In [ ]:
# You can explicitly clear the queue if needed
remaining = antenna.clear_queue()

print(f"Cleared {remaining} frames from queue")
print(f"Queue content after: {antenna.queue_content} frames")

## Demonstrating Frame Accumulation

**NEW in v4.0**: Frames accumulate between multiple `run()` calls!

This gives you flexibility to combine multiple acquisitions or start fresh with `clear_queue()`.

In [ ]:
# First acquisition: 0.3 seconds
antenna.run(mems=[0, 1, 2], duration=0.3, sampling_frequency=44100, frame_length=1024)
antenna.wait()
frames_after_run1 = antenna.queue_content
print(f"After run 1 (0.3s): {frames_after_run1} frames")

# Second acquisition WITHOUT clearing → frames accumulate!
antenna.run(mems=[0, 1, 2], duration=0.3, sampling_frequency=44100, frame_length=1024)
antenna.wait()
frames_after_run2 = antenna.queue_content
print(f"After run 2 (0.3s): {frames_after_run2} frames")
print(f"✨ Accumulated {frames_after_run2 - frames_after_run1} new frames!")

# Consume some frames
consumed = 5
for i, frame in enumerate(antenna):
    if i >= consumed - 1:
        break
print(f"After consuming {consumed} frames: {antenna.queue_content} frames remain")

# Clear all for next example
cleared = antenna.clear_queue()
print(f"Cleared {cleared} remaining frames → Queue: {antenna.queue_content}")

## Using All Available MEMS

You can activate all available MEMS using the `available_mems` property.

In [ ]:
antenna.run(
    duration=0.5,                          # 0.5 second acquisition  
    mems=antenna.available_mems,           # All available MEMS
    counter=False,                         # No counter channel
    datatype='int32',                      # int32 (default)
    sampling_frequency=50000               # 50 kHz
)

antenna.wait()

print(f"✓ Acquired with {len(antenna.mems)} MEMS at {antenna.sampling_frequency} Hz")
print(f"✓ Frames available: {antenna.queue_content}")

# Clear queue for next example
antenna.clear_queue()

In [ ]:
# You can also selectively activate specific MEMS
antenna.run(
    mems=[0, 10, 20],  # Only MEMS 0, 10, and 20
    duration=0.5
)

antenna.wait()

print(f"✓ Active MEMS: {antenna.mems}")

# Clear queue for next example
antenna.clear_queue()

## Real-Time Processing

The `run()` method is **non-blocking**! You can iterate over frames as they arrive.

**Critical**: Always call `wait()` after `run()` to ensure proper cleanup!

**Note**: In v4.0, if you call `run()` while a previous acquisition is still active, it will automatically stop the previous one and start fresh. No manual cleanup needed!

In [ ]:
log.setLevel("WARNING")  # Reduce logs

antenna.run(
    duration=1,
    mems=[0, 1, 2, 3],
    counter=True,        # Enable counter channel
    frame_length=1024
)

# Process frames in real-time
frame_count = 0
for frame in antenna:
    # frame[0] = counter channel
    # frame[1:] = MEMS data
    counter_value = frame[0, 0]
    print(f"Processing frame #{counter_value:05.0f} - Energy: {np.mean(frame[1:]**2):.2e}")
    frame_count += 1

antenna.wait()  # ← ALWAYS call wait()!

print(f"\n✓ Processed {frame_count} frames in real-time")
print(f"✓ Queue now empty: {antenna.queue_content} frames")

## Measuring Acquisition Time

Let's measure the actual time for a 1-second acquisition.

**Note**: The total time includes the queue timeout (default 1 second), so expect ~2 seconds total.

In [ ]:
start_time = time.time()

antenna.run(
    duration=1,
    mems=[0, 1],
    counter=True,
    queue_timeout=1000  # 1000ms timeout (default)
)

# Collect counter values
counter_values = []
for frame in antenna:
    counter_values.append(int(frame[0, 0]))

antenna.wait()

end_time = time.time()
elapsed = end_time - start_time

print(f"Requested duration: 1.0 seconds")
print(f"Actual elapsed time: {elapsed:.2f} seconds")
print(f"Frames received: {len(counter_values)}")
print(f"Counter range: {min(counter_values)} → {max(counter_values)}")
print(f"\n💡 Elapsed time = acquisition (1s) + queue timeout (1s) = ~2s")

## Understanding Queue Timeout

The queue timeout is the maximum time to wait for new frames.

**Total time** = Acquisition duration + Queue timeout

You can adjust the timeout if needed:

In [ ]:
# Example with shorter timeout
start_time = time.time()

antenna.run(
    duration=0.5,
    mems=[0, 1],
    queue_timeout=200  # 200ms timeout (faster!)
)

frame_count = sum(1 for _ in antenna)
antenna.wait()

elapsed = time.time() - start_time

print(f"Duration: 0.5s, Timeout: 200ms")
print(f"Elapsed: {elapsed:.2f}s (~0.7s expected)")
print(f"Frames: {frame_count}")

## NEW in v4.0: Configuration Objects

For complex configurations, use the type-safe `AcquisitionConfig` class:

In [ ]:
# Create a reusable configuration
config = AcquisitionConfig(
    mems=[0, 1, 2, 3, 4, 5, 6, 7],
    sampling_frequency=50000,
    frame_length=2048,
    duration=2.0,
    datatype='float32',  # Automatically converts to Pascals!
    counter=True,
    queue_size=50
)

# Check computed properties
print(f"Configuration:")
print(f"  Total samples: {config.total_samples}")
print(f"  Total frames: {config.total_frames}")
print(f"  Active channels: {len(config.active_channels)}")

# Use the configuration
antenna.run(**config.__dict__)

# Collect all data
all_frames = []
for frame in antenna:
    all_frames.append(frame)

antenna.wait()

# Concatenate frames
signal = np.concatenate(all_frames, axis=1)
print(f"\n✓ Collected signal: {signal.shape}")
print(f"✓ Data type: {signal.dtype} (float32 = Pascals)")

## MEMS Positions and Geometry

You can define the physical positions of MEMS in 3D space for beamforming applications.

Example: Circular 32-MEMS antenna with 350mm diameter

In [ ]:
from megamicros.geometry import circle
from megamicros import MemsArrayInfo

# Define circular antenna geometry
mems_number = 32
antenna_radius = 0.175  # meters
antenna_height = 0.0
antenna_angle_offset = 2 * np.pi / mems_number / 2

mems_positions = np.array(circle(
    points_number=mems_number,
    radius=antenna_radius,
    height=antenna_height,
    angle_offset=antenna_angle_offset,
    clockwise=True
))

print(f"MEMS positions shape: {mems_positions.shape}")
print(f"First 3 MEMS positions (x, y, z) in meters:")
print(mems_positions[:3])

# Visualize geometry
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(mems_positions[:, 0], mems_positions[:, 1], s=100, c='blue', marker='o')
ax.set_xlabel('X (meters)')
ax.set_ylabel('Y (meters)')
ax.set_title('32-MEMS Circular Antenna Geometry')
ax.grid(True)
ax.axis('equal')
plt.show()

## Next Steps

- **Notebook 02**: USB interface details
- **Notebook 03**: Advanced Megamicros features  
- **Notebook 04**: Beamforming with FDAS algorithm
- **Notebook 06**: Working with H5 files

## Key Takeaways v4.0

✅ `Megamicros()` auto-detects data source (USB, H5, or Random)  
✅ **Simple pattern**: `run()` → process frames → `wait()`  
✅ **Auto-cleanup**: Re-running `run()` automatically stops previous acquisition  
✅ **Frame accumulation**: Frames from multiple `run()` calls accumulate in queue (unless `queue_size` changes)  
✅ Use `clear_queue()` to explicitly discard frames when you want a fresh start  
✅ Use `AcquisitionConfig` for type-safe, validated configurations  
✅ `RandomDataSource` enables testing without hardware  
✅ All v3.x code still works unchanged  

**See `USER_GUIDE_v4.md` for complete documentation!**